<a href="https://colab.research.google.com/github/pydeoxy/cic_ai_course/blob/main/langchain_llama3_2_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

More about LangChain RAG:

https://learn.deeplearning.ai/courses/langchain-chat-with-your-data/

https://docs.langchain.com/oss/python/langchain/rag


## Dependencies

pip install transformers langchain langchain-huggingface langchain-community faiss-cpu

In [ ]:
!pip install langchain langchain-huggingface langchain-community faiss-cpu

In [ ]:
# Import dependencies
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
    )
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import BeautifulSoupTransformer
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Turn on line wrapping
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
      white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)

## Load LLM and simple QA



In [ ]:
# Load LLM
model_id = "unsloth/Llama-3.2-1B-Instruct"
# microsoft/Phi-4-mini-instruct
# google/gemma-3-1b-it
# meta-llama/Llama-3.2-1B-Instruct
# unsloth/Llama-3.2-1B-Instruct

# LLM
llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)


# Chat
chat = ChatHuggingFace(llm=llm, verbose=True)

In [ ]:
llm.invoke("What is SmartLab?")

In [ ]:
messages = [
    ("system", "You are a helpful assistant."),
    ("human", "What is SmartLab?"),
]

chat.invoke(messages)

# RAG from webpage

In [ ]:
# Load documents from webpage
urls = ["https://www.metropolia.fi/fi/tutkimus-kehitys-ja-innovaatiot/yhteistyoalustat/smart-lab"]
loader_html = AsyncHtmlLoader(urls)
docs_html = loader_html.load()

bs_transformer = BeautifulSoupTransformer()
docs_transformed = bs_transformer.transform_documents(docs_html, tags_to_extract=["div"])

# Split it into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs_transformed_split = text_splitter.split_documents(docs_transformed)


print(len(docs_transformed_split))
print(len(docs_transformed[0].page_content))

In [ ]:
# create the open-source embedding function
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# load it into FAISS
html_db = FAISS.from_documents(docs_transformed_split, embedding_function)

# Retrival QA

In [ ]:
# Retriever
retriever_html = html_db.as_retriever()


# 1. Define the 'Answering' logic (The "Stuff" Chain)
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

# This creates a chain that takes 'context' and 'input' and sends them to the LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# 2. Define the 'Retrieval' logic
# This connects your vector store (retriever) to the answering chain
rag_chain = create_retrieval_chain(retriever_html, question_answer_chain)

# 3. Execution
# Note: The input key is now "input", and the answer is in "answer"
response = rag_chain.invoke({"input": "What is SmartLab?"})

print(response["answer"])

In [ ]:
# Clear RAM cache
#del pipe

# Clear CUDA cache
import torch
torch.cuda.empty_cache()